### Compact Form (Kassimali)

It is often convenient to write Eq. (22) in the compact form

$$
\mathbf{K}_t = \left(\frac{EA}{L}\right)\mathbf{T}^T\mathbf{T} + q\,\mathbf{G}
\tag{23}
$$

where the **geometric matrix** $\mathbf{G}$ is

$$
\mathbf{G}
=
\frac{1}{\bar{L}}
\begin{bmatrix}
c_y^2 & -c_x\!c_y & -c_y^2 & c_x\!c_y\\
-c_x\!c_y & c_x^2 & c_x\!c_y & -c_x^2\\
-c_y^2 & c_x\!c_y & c_y^2 & -c_x\!c_y\\
c_x\!c_y & -c_x^2 & -c_x\!c_y & c_x^2
\end{bmatrix}
$$

This form separates the tangent stiffness into a **material part** and a **geometric part**.

### Compact Form (Veenendaal)

Veenendaal writes the geometric term in the compact form

$$
\mathbf{K}_t
=
\left(\frac{EA}{L}\right)\mathbf{T}^T\mathbf{T}
+
\frac{q}{\bar{L}}\left(\mathbf{I}-\mathbf{T}^T\mathbf{T}\right)
\tag{24}
$$

This comparison is useful, but it is important to note the following:

- this form works cleanly for **node-level stiffness blocks**
- in **2D**, that means a **$2\times2$** block at a node
- in **3D**, that means a **$3\times3$** block at a node
- it does **not** directly reproduce Kassimali’s **full $4\times4$ 2D truss-element** geometric stiffness matrix

So the Veenendaal form and the Kassimali form are directly comparable only when we isolate the stiffness contribution at a **single node**, not when we look at the **full two-node element matrix**.

### Why the Full $4\times4$ Comparison Does Not Match

For the full 2D truss element,

$$
\mathbf{T}
=
\begin{bmatrix}
-c_x & -c_y & c_x & c_y
\end{bmatrix}
$$

which gives

$$
\mathbf{T}^T\mathbf{T}
=
\left[
\begin{smallmatrix}
c_x^2 & c_xc_y & -c_x^2 & -c_xc_y\\
c_xc_y & c_y^2 & -c_xc_y & -c_y^2\\
-c_x^2 & -c_xc_y & c_x^2 & c_xc_y\\
-c_xc_y & -c_y^2 & c_xc_y & c_y^2
\end{smallmatrix}
\right]
$$

From Veenendaal:

and therefore, using the identity $c_x^2 + c_y^2 = 1$,

$$
\frac{q}{\bar{L}}\left(\mathbf{I}-\mathbf{T}^T\mathbf{T}\right)
=
\frac{q}{\bar{L}}
\left[
\begin{smallmatrix}
c_y^2 & -c_xc_y & c_x^2 & c_xc_y\\
-c_xc_y & c_x^2 & c_xc_y & c_y^2\\
c_x^2 & c_xc_y & c_y^2 & -c_xc_y\\
c_xc_y & c_y^2 & -c_xc_y & c_x^2
\end{smallmatrix}
\right]
$$

But Kassimali’s geometric matrix for the full $4\times4$ element is

$$
q\,\mathbf{G}
=
\frac{q}{\bar{L}}
\left[
\begin{smallmatrix}
c_y^2 & -c_xc_y & \boxed{-c_y^2} & \boxed{c_xc_y}\\
-c_xc_y & c_x^2 & \boxed{c_xc_y} & \boxed{-c_x^2}\\
\boxed{-c_y^2} & \boxed{c_xc_y} & c_y^2 & -c_xc_y\\
\boxed{c_xc_y} & \boxed{-c_x^2} & -c_xc_y & c_x^2
\end{smallmatrix}
\right]
$$
So the matrices are **not the same**.

The top-left and bottom-right are MATCHING

The mismatch occurs in the **off-diagonal $2\times2$ blocks** (HIGHLIGHTED), which means:

- the Veenendaal expression is **not** a direct replacement for Kassimali’s full $4\times4$ geometric matrix
- the agreement only appears when we compare **single-node blocks** rather than the **full element matrix**

In [1]:
import numpy as np

# -----------------------------------------------------------------------------
# Notes:
#
# These little functions are just me testing the relationship between two
# different ways of writing the geometric stiffness term that appear in two
# different references:
#
# 1. In Veenendaal, the geometric term is written in the form
#
#       (q / Lbar) * (I - T.T @ T)
#
# 2. In Kassimali, the geometric term is written in the form
#
#       (q / Lbar) * G
#
# The important point is that these two references are not writing stiffness
# at the same "level":
#
# - Kassimali is writing the geometric stiffness for the FULL 2D truss element,
#   which is a 4x4 matrix involving both end nodes of the member.
#
# - Veenendaal is writing stiffness at the NODE / BRANCH level, so the matrix
#   there is a smaller directional stiffness block.
#
# Because of that, the Veenendaal form
#
#       I - T.T @ T
#
# does NOT translate directly to the full 4x4 Kassimali element matrix.
#
# However, if I look only at a SINGLE NODE, then the two forms do match:
#
# - in 2D, the 2x2 single-node Veenendaal form is equal to the corresponding
#   2x2 Kassimali-style G block
#
# - in 3D, the 3x3 single-node Veenendaal form gives the corresponding 3x3
#   nodal geometric stiffness block
#
# So:
#
# - full 4x4 element level -> Kassimali G is NOT equal to I - T.T @ T
# - single-node level      -> Kassimali-style G DOES match I - T.T @ T
#
# These functions are just checking that idea explicitly.
# -----------------------------------------------------------------------------



In [2]:

# -----------------------------------------------------------------------------
# FULL 2D TRUSS ELEMENT (4x4)
#
# This is the full Kassimali-style 2D element geometric stiffness term.
#
# Here the matrix includes both nodes of the element, so it is a 4x4 matrix.
# Because this is a full element matrix, it cannot be replaced by the
# Veenendaal-style expression
#
#     I - T.T @ T
#
# using T = [-cx, -cy, cx, cy]
#
# In other words, the Veenendaal expression does not carry over directly from
# node-level stiffness to the full 2D truss element stiffness.
# -----------------------------------------------------------------------------
def truss_tangent_stiffness_4x4(Lbar, cx, cy, q):
    T = np.array([[-cx, -cy, cx, cy]])

    g = np.array([
        [cy**2,  -cx*cy,   -cy**2,  cx*cy],
        [-cx*cy,  cx**2,    cx*cy, -cx**2],
        [-cy**2,  cx*cy,    cy**2, -cx*cy],
        [cx*cy,  -cx**2,   -cx*cy,  cx**2],
    ])

    # Kassimali-style full 4x4 element geometric stiffness term
    kg_kassimali = q / Lbar * g

    # Veenendaal-style expression tested on the full 4x4 element
    # This does NOT match kt1
    kg_veenendaal = q / Lbar * (np.identity(4) - T.T @ T)
    print("kg_kassimali = Kassimali-style full 4x4 geometric stiffness")
    print(kg_kassimali)
    print("\nkg_veenendaal = Veenendaal-style I - T.T @ T applied to full 4x4 element")
    print(kg_veenendaal)



In [3]:

# -----------------------------------------------------------------------------
# SINGLE NODE IN 2D (2x2)
#
# If I isolate just one node in 2D, then the Kassimali-style G block becomes
# a 2x2 matrix:
#
#     G = [[ cy^2,   -cx*cy],
#          [-cx*cy,   cx^2 ]]
#
# At this single-node level, this IS equal to
#
#     I - T.T @ T
#
# with
#
#     T = [-cx, -cy]
#
# So this is the clean place where the Veenendaal form and the Kassimali-style
# G matrix agree.
# -----------------------------------------------------------------------------
def truss_tangent_stiffness_2x2(Lbar, cx, cy, q):
    T = np.array([[-cx, -cy]])

    g = np.array([
        [cy**2,  -cx*cy],
        [-cx*cy,  cx**2],
    ])

    # Kassimali-style single-node 2x2 block
    kg_kassimali = q / Lbar * g

    # Veenendaal-style form
    kg_veenendaal = q / Lbar * (np.identity(2) - T.T @ T)

    print("kg_kassimali = Kassimali-style 2x2 nodal geometric block")
    print(kg_kassimali)
    print("\nkg_veenendaal = Veenendaal-style 2x2 form, should match exactly")
    print(kg_veenendaal)


In [4]:

# -----------------------------------------------------------------------------
# SINGLE NODE IN 3D (3x3)
#
# The same idea extends to a single node in 3D.
#
# If I define
#
#     T = [-cx, -cy, -cz]
#
# then the Veenendaal-style expression
#
#     I - T.T @ T
#
# gives the 3x3 nodal geometric stiffness block.
#
# So again, at the NODE level, this form works cleanly.
#
# This is different from the full element case, where a larger matrix with
# both nodes is involved.
#
# One additional thing I tested here was the planar case:
#
# - if the bar lies in the x-y plane, then cz = 0
# - in that case, I - T.T @ T gives a nonzero value in the (3,3) position
#
# At first this seemed wrong to me, because for the usual ELASTIC stiffness
# term, there would be no axial stiffness contribution in z for a bar lying
# in the x-y plane.
#
# But this is not the elastic/material stiffness term.
# This is the GEOMETRIC stiffness term.
#
# For geometric stiffness, a direction can still contribute even if it is not
# along the bar axis. The z-direction is transverse to a bar lying in the
# x-y plane, so an axially loaded bar can develop geometric stiffness against
# small z-direction perturbations.
#
# So:
# - for elastic stiffness, z would not appear here as an axial stiffness term
# - for geometric stiffness, z can still appear because the bar is already
#   carrying axial force, and small transverse motions produce second-order
#   effects
#
# That is why the (3,3) entry is not zero in the planar 3D test.
# -----------------------------------------------------------------------------
def truss_tangent_stiffness_3x3(Lbar, cx, cy, cz, q):
    T = np.array([[-cx, -cy, -cz]])

    g = np.array([
        [1 - cx**2,  -cx*cy,    -cx*cz],
        [ -cx*cy,   1 - cy**2,  -cy*cz],
        [ -cx*cz,    -cy*cz,   1 - cz**2],
    ])

    # 3D nodal geometric stiffness block written directly
    kg_kassimali = q / Lbar * g

    # Veenendaal-style form
    kg_veenendaal = q / Lbar * (np.identity(3) - T.T @ T)

    print("kg_kassimali = direct 3x3 nodal geometric block")
    print(kg_kassimali)
    print("\nkg_veenendaal = Veenendaal-style 3x3 form, should match exactly")
    print(kg_veenendaal)

In [5]:
# ---------------------------------------------------------------------
# Test values
# ---------------------------------------------------------------------
Lbar = 5000.0
cx, cy, cz = 0.8660254037844386, 0.5, 0.0
q = 15000.0

print("Input values:")
print(f"  Lbar = {Lbar}")
print(f"  cx   = {cx}")
print(f"  cy   = {cy}")
print(f"  cz   = {cz}")
print(f"  q    = {q}")

# ---------------------------------------------------------------------
# Run comparison tests
# ---------------------------------------------------------------------
print("\n" + "="*72)
print("TEST 1: Full 2D truss element geometric stiffness (4x4)")
print("="*72)
truss_tangent_stiffness_4x4(Lbar, cx, cy, q)

print("\n" + "="*72)
print("TEST 2: Single-node 2D geometric stiffness block (2x2)")
print("="*72)
truss_tangent_stiffness_2x2(Lbar, cx, cy, q)

print("\n" + "="*72)
print("TEST 3: Single-node 3D geometric stiffness block (3x3)")
print("="*72)
truss_tangent_stiffness_3x3(Lbar, cx, cy, cz, q)

Input values:
  Lbar = 5000.0
  cx   = 0.8660254037844386
  cy   = 0.5
  cz   = 0.0
  q    = 15000.0

TEST 1: Full 2D truss element geometric stiffness (4x4)
kg_kassimali = Kassimali-style full 4x4 geometric stiffness
[[ 0.75       -1.29903811 -0.75        1.29903811]
 [-1.29903811  2.25        1.29903811 -2.25      ]
 [-0.75        1.29903811  0.75       -1.29903811]
 [ 1.29903811 -2.25       -1.29903811  2.25      ]]

kg_veenendaal = Veenendaal-style I - T.T @ T applied to full 4x4 element
[[ 0.75       -1.29903811  2.25        1.29903811]
 [-1.29903811  2.25        1.29903811  0.75      ]
 [ 2.25        1.29903811  0.75       -1.29903811]
 [ 1.29903811  0.75       -1.29903811  2.25      ]]

TEST 2: Single-node 2D geometric stiffness block (2x2)
kg_kassimali = Kassimali-style 2x2 nodal geometric block
[[ 0.75       -1.29903811]
 [-1.29903811  2.25      ]]

kg_veenendaal = Veenendaal-style 2x2 form, should match exactly
[[ 0.75       -1.29903811]
 [-1.29903811  2.25      ]]

TEST 3: S